# Robin transport and transient heat

The scalar RAD local form is skew in advection; its multiplier is $(-K\nabla u+\beta u/2)\cdot n$. The heat integrator uses backward Euler with time-dependent source and Dirichlet data.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
import inspect
import numpy as np
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace, assemble
from examples.formulations.scalar import (
    ScalarDiscretization, transport_equations, transport_problem,
    heat_equations, heat_problem, initial_coefficients, recover_scalar,
)

# These declarations produce the blocks consumed by the shared core.
print(inspect.getsource(transport_equations))
print(inspect.getsource(heat_equations))


In [ ]:
mesh = TriangleMesh.unit_square(2)
space = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces))

def scalar(points):
    """Return the exact affine scalar field."""
    return 1 + points[:, 0] + 2 * points[:, 1]

data = ScalarDiscretization(mesh, space)
problem = transport_problem(
    data, velocity=(1.0, 0.5), reaction=2.0,
    source=lambda points: 2 * scalar(points) + 2, dirichlet=scalar,
)
system = assemble(problem)
rad = recover_scalar(data, system.solve(), system.local_metadata)
assert rad.l2_error(scalar) < 1e-10

# Each time step declares M/dt + K and the original M*old/dt history load.
heat_data = ScalarDiscretization(mesh, SkeletonSpace(mesh))
previous = initial_coefficients(heat_data, scalar)
times = [0.0, 0.01, 0.03, 0.1]
solutions = []
for old_time, time in zip(times[:-1], times[1:], strict=True):
    problem = heat_problem(
        heat_data, previous, time - old_time,
        source=lambda points: np.ones(len(points)),
        dirichlet=lambda points, time=time: scalar(points) + time,
    )
    system = assemble(problem)
    coefficients = system.solve()
    result = recover_scalar(heat_data, coefficients, system.local_metadata)
    previous = coefficients.fields
    solutions.append(result)
    error = result.l2_error(lambda points, time=time: scalar(points) + time)
    assert error < 1e-10
    print("time", time, "space-time patch error", error)


The stationary local forms are declared in the application, with skew advection and a retained constant amplitude. The velocity here is constant and divergence-free; this example does not declare SUPG or UNUSUAL stabilization.

Each backward-Euler step writes

$$
\left(K\nabla u^{n+1},\nabla v\right)
+\frac{\left(u^{n+1},v\right)}{\Delta t}
+\langle\lambda^{n+1},v\rangle
=\left(f^{n+1},v\right)
+\frac{\left(u^n,v\right)}{\Delta t}.
$$

Backward Euler is first order in time for general data. The affine space-time patch checks signs and history; the automated temporal refinement tests use nonaffine evolution. Both steps call the same variational assembly and solution functions used for other operators.
